# Flux Dev Photo Generator — Colab T4

**Портретные фото Instagram-качества** на базе Flux Dev fp8 + IPAdapter FaceID + перенос стиля.

---

### Что делает этот ноутбук?

Генерирует **реалистичные портретные фотографии** с сохранением вашего лица.
Вы загружаете несколько референсных фото — нейросеть извлекает черты лица (FaceID)
и переносит их на новые изображения в стиле Instagram-фотосессии.

**Результат:** профессиональные портреты 864x1080 с вашим лицом в разных локациях и стилях.

---

### Пайплайн

```
[1. GPU] → [2. Установка] → [3. HF Токен] → [4. Модели] → [5. Воркфлоу] → [6. ЗАГРУЗКА ФОТО] → [7. Запуск]
   │           │                  │               │              │                  │                    │
   │           │                  │               │              │                  │                    └─ Открыть ссылку,
   │           │                  │               │              │                  │                       генерировать
   │           │                  │               │              │                  │
   │           │                  │               │              │                  └─ Загрузить 3-5 фото
   │           │                  │               │              │                     вашего лица
   │           │                  │               │              │
   │           │                  │               │              └─ 2 воркфлоу:
   │           │                  │               │                 FaceID + img2img
   │           │                  │               │
   │           │                  │               └─ Flux Dev fp8, CLIP Vision,
   │           │                  │                  IPAdapter (~18 ГБ)
   │           │                  │
   │           │                  └─ Авторизация HF
   │           │                     (нужна для Flux)
   │           │
   │           └─ ComfyUI + IPAdapter +
   │              KJNodes + InsightFace
   │
   └─ Проверка T4 / VRAM
```

---

### Воркфлоу

| Воркфлоу | Вход | Выход |
|----------|------|-------|
| `photo_flux_instagram` | Референсные фото лица + текст | Портреты с консистентным FaceID |
| `photo_flux_img2img` | Референсное изображение + текст | Перенос стиля/композиции |

---

### Какие фото нужны?

Для качественного FaceID подготовьте **3-5 референсных фото** вашего лица:

| Требование | Хорошо | Плохо |
|-----------|--------|-------|
| **Освещение** | Равномерное, дневное | Тёмное, контрастное |
| **Ракурс** | Прямо + лёгкий поворот | Сильный профиль, наклон |
| **Лицо** | Полностью видно, без очков | Закрыто волосами/руками |
| **Качество** | Чёткое, в фокусе | Размытое, сжатое |
| **Формат** | PNG или JPG, от 512px | Маленькие иконки, GIF |
| **Выражение** | Нейтральное или лёгкая улыбка | Гримасы, закрытые глаза |

> **Совет:** Разные ракурсы (анфас + 3/4) дают лучший результат, чем 5 одинаковых селфи.

**Запускайте ячейки 1-7 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0 — предотвращает конфликты с предустановленными пакетами Colab
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI_IPAdapter_plus || git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git
# ControlNet preprocessors (depth, canny, pose extraction)
!test -d comfyui_controlnet_aux || git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git

!pip install insightface onnxruntime -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI_IPAdapter_plus/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r comfyui_controlnet_aux/requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nУстановлено!")

In [ ]:
#@title 3. Авторизация Hugging Face (обязательно для FLUX)
#@markdown ---
#@markdown ### Зачем это нужно?
#@markdown Модель **Flux Dev fp8** находится в закрытом репозитории на Hugging Face.
#@markdown Для скачивания нужно:
#@markdown 1. Зарегистрироваться на [huggingface.co](https://huggingface.co)
#@markdown 2. Принять лицензию на странице: [Comfy-Org/flux1-dev](https://huggingface.co/Comfy-Org/flux1-dev)
#@markdown 3. Создать токен: [Settings → Access Tokens](https://huggingface.co/settings/tokens) (тип: **Read**)
#@markdown 4. Вставить токен ниже

hf_token = "" #@param {type:"string"}
#@markdown Вставьте ваш Hugging Face токен (начинается с `hf_...`)

if not hf_token or not hf_token.strip().startswith("hf_"):
    print("=" * 60)
    print("  ОШИБКА: Токен не указан или неверный формат!")
    print()
    print("  1. Перейдите: https://huggingface.co/settings/tokens")
    print("  2. Создайте токен с доступом Read")
    print("  3. Вставьте его в поле hf_token выше")
    print("  4. Перезапустите эту ячейку")
    print("=" * 60)
    raise ValueError("Hugging Face токен обязателен для Flux Dev")

!pip install huggingface_hub -q
from huggingface_hub import login
login(token=hf_token.strip())

import os
os.environ["HF_TOKEN"] = hf_token.strip()

print("\n" + "=" * 60)
print("  Авторизация успешна!")
print("  Теперь можно скачивать модели Flux Dev")
print("=" * 60)

In [ ]:
#@title 4. Скачивание моделей (~21 ГБ)
#@markdown Модель Flux Dev fp8 требует авторизации HF (ячейка 3).
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/checkpoints", exist_ok=True)
os.makedirs(f"{M}/clip_vision", exist_ok=True)
os.makedirs(f"{M}/ipadapter", exist_ok=True)
os.makedirs(f"{M}/controlnet", exist_ok=True)
os.makedirs(f"{M}/ultralytics/bbox", exist_ok=True)

hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("HF_TOKEN не найден! Запустите ячейку 3 (Авторизация HF) заново.")
    raise ValueError("HF_TOKEN не установлен — необходим для скачивания Flux Dev fp8")

# Файлы, требующие авторизации HF (gated repos)
gated_files = {
    # Flux Dev fp8 (~11.5 ГБ) — gated repo
    f"{M}/checkpoints/flux1-dev-fp8.safetensors":
        "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
}

# Файлы, не требующие авторизации (открытые репозитории)
open_files = {
    # CLIP Vision H14 (~3.9 ГБ)
    f"{M}/clip_vision/model.safetensors":
        "https://huggingface.co/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/model.safetensors",
    # IPAdapter FaceID PlusV2 SDXL (~0.9 ГБ)
    f"{M}/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin":
        "https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin",
    # IPAdapter Flux (перенос стиля) (~5.3 ГБ)
    f"{M}/ipadapter/ip-adapter_flux.bin":
        "https://huggingface.co/InstantX/FLUX.1-dev-IP-Adapter/resolve/main/ip-adapter.bin",
    # ControlNet Depth для Flux (~3.6 ГБ) — depth matching из референса
    f"{M}/controlnet/flux-dev-controlnet-depth.safetensors":
        "https://huggingface.co/Shakker-Labs/FLUX.1-dev-ControlNet-Depth/resolve/main/diffusion_pytorch_model.safetensors",
    # Face detection model для FaceDetailer (~6 МБ)
    f"{M}/ultralytics/bbox/face_yolov8m.pt":
        "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt",
}

# Скачиваем gated модели с авторизацией
for dest, url in gated_files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)} (с авторизацией HF)...")
        !wget -q --show-progress --header="Authorization: Bearer {hf_token}" -O "{dest}" "{url}"
        if os.path.exists(dest) and os.path.getsize(dest) < 1000:
            os.remove(dest)
            print(f"  Ошибка скачивания! Проверьте, что вы приняли лицензию:")
            print(f"  https://huggingface.co/Comfy-Org/flux1-dev")
            raise RuntimeError("Не удалось скачать Flux Dev fp8. Примите лицензию на HuggingFace.")

# Скачиваем открытые модели
for dest, url in open_files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания открытых моделей
download_errors = []
for dest in open_files:
    if not os.path.exists(dest) or os.path.getsize(dest) < 1024:
        if os.path.exists(dest):
            os.remove(dest)
        download_errors.append(os.path.basename(dest))
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

if download_errors:
    print(f"\n{'='*60}")
    print(f"  НЕ УДАЛОСЬ СКАЧАТЬ {len(download_errors)} файл(ов):")
    for name in download_errors:
        print(f"    - {name}")
    print(f"{'='*60}")
    raise RuntimeError(f"Не скачаны модели: {', '.join(download_errors)}. Перезапустите ячейку.")

print("\nВсе модели готовы!")
!du -sh {M}/checkpoints/* {M}/clip_vision/* {M}/ipadapter/* {M}/controlnet/* {M}/ultralytics/bbox/*

In [ ]:
#@title 5. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

# Скачиваем Colab-версию (с исправленными путями)
!wget -q -O "{WF_DIR}/photo_flux_instagram.json" "{RAW}/photo_flux_instagram_colab.json"
print("Скачано: photo_flux_instagram (пути для Colab)")

# Скачиваем воркфлоу img2img
!wget -q -O "{WF_DIR}/photo_flux_img2img.json" "{RAW}/photo_flux_img2img.json"
print("Скачано: photo_flux_img2img")

print(f"\n2 воркфлоу сохранены в {WF_DIR}")

In [ ]:
#@title 6. Загрузка референсных фото лица
#@markdown ---
#@markdown ### Что это?
#@markdown Это **референсные фото вашего лица** для технологии FaceID.
#@markdown Нейросеть извлечёт черты лица и перенесёт их на сгенерированные портреты.
#@markdown Это **НЕ обучение** — фото используются только как образец.
#@markdown
#@markdown ---
#@markdown ### 3 папки — 3 типа локации
#@markdown | Папка | Назначение | Пример |
#@markdown |-------|-----------|--------|
#@markdown | `portrait` | Крупный план, студийные | Селфи, фото на документы |
#@markdown | `hall` | Интерьер, помещения | Фото в кафе, дома, в офисе |
#@markdown | `street` | Улица, экстерьер | Фото на прогулке, в парке |
#@markdown
#@markdown Воркфлоу использует **ImageMaskSwitch** для переключения между папками.
#@markdown Можно загрузить фото только в `portrait` — этого достаточно для старта.
#@markdown
#@markdown ---
#@markdown ### Выберите папку для загрузки:

from google.colab import files
from IPython.display import display, Image as IPImage
import os

ALLOWED_FORMATS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}

INPUT = "/content/ComfyUI/input"
for folder in ["refs/portrait", "refs/hall", "refs/street"]:
    os.makedirs(f"{INPUT}/{folder}", exist_ok=True)

target = "refs/portrait" #@param ["refs/portrait", "refs/hall", "refs/street"]

print("=" * 60)
print(f"  ЗАГРУЗКА РЕФЕРЕНСНЫХ ФОТО")
print(f"  Папка: {target}")
print(f"  Допустимые форматы: {', '.join(sorted(ALLOWED_FORMATS))}")
print("=" * 60)
print()

uploaded = files.upload()

saved_count = 0
skipped = []

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext not in ALLOWED_FORMATS:
        skipped.append(f"  [ПРОПУЩЕН] {name} — формат '{ext}' не поддерживается")
        continue

    path = os.path.join(INPUT, target, name)
    with open(path, "wb") as f:
        f.write(data)

    size_kb = len(data) / 1024
    saved_count += 1
    print(f"  [OK] {name} ({size_kb:.0f} КБ) -> {target}/")

if skipped:
    print()
    print("Пропущенные файлы (неверный формат):")
    for s in skipped:
        print(s)
    print(f"Допустимые форматы: {', '.join(sorted(ALLOWED_FORMATS))}")

print()
print("-" * 60)
print(f"  Сохранено: {saved_count} файл(ов) в {target}")
print("-" * 60)

# Превью загруженных фото
target_path = os.path.join(INPUT, target)
all_files = [f for f in os.listdir(target_path)
             if os.path.splitext(f)[1].lower() in ALLOWED_FORMATS]

if all_files:
    print(f"\n  Все фото в папке {target} ({len(all_files)} шт.):")
    print()
    for img_name in sorted(all_files):
        img_path = os.path.join(target_path, img_name)
        print(f"  {img_name}")
        try:
            display(IPImage(filename=img_path, width=200))
        except Exception:
            print(f"    (превью недоступно)")
    print()

# Итоговая сводка по всем папкам
print("=" * 60)
print("  СВОДКА ПО ВСЕМ ПАПКАМ:")
for folder in ["refs/portrait", "refs/hall", "refs/street"]:
    folder_path = os.path.join(INPUT, folder)
    folder_files = [f for f in os.listdir(folder_path)
                    if os.path.splitext(f)[1].lower() in ALLOWED_FORMATS]
    status = f"{len(folder_files)} фото" if folder_files else "пусто"
    marker = " <-- текущая" if folder == target else ""
    print(f"  {folder}: {status}{marker}")
print("=" * 60)

In [ ]:
#@title 7. Запуск ComfyUI
import subprocess, time, re, os, requests

# Борьба с фрагментацией CUDA памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
for i in range(12):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            break
    except Exception:
        pass
else:
    print("  ComfyUI не ответил за 60 сек — проверьте логи выше.")

# Cloudflare туннель
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI готов! Открывайте: {url}")
    print(f"{'='*60}")
    print("\nЗагрузите воркфлоу: Меню -> Load -> photo_flux_instagram")
else:
    print("Ссылка на туннель не найдена. ComfyUI работает на порту 8188.")

In [ ]:
#@title 8. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/flux"
os.makedirs(dst, exist_ok=True)

images = glob.glob(f"{src}/*.png") + glob.glob(f"{src}/*.jpg")
if not images:
    print("Результатов пока нет. Сначала сгенерируйте изображения!")
else:
    for img in images:
        shutil.copy2(img, dst)
        print(f"Скопировано: {os.path.basename(img)}")
    print(f"\n{len(images)} файлов сохранено в {dst}")

---
## Инструкция по использованию

### Общий порядок работы

```
Открыть ссылку → Загрузить воркфлоу → Настроить промпт → Queue Prompt → Результат в output/
```

---

### Воркфлоу 1: `photo_flux_instagram` — FaceID портреты

Генерирует портреты с **вашим лицом** в разных стилях и локациях.

**Пошагово:**
1. Откройте ссылку на туннель из ячейки 7
2. Меню **Load** -> выберите **photo_flux_instagram**
3. Воркфлоу автоматически подхватит фото из папок `refs/`
4. Выберите папку через **ImageMaskSwitch**: `1` = portrait, `2` = hall, `3` = street
5. Отредактируйте промпт (описание желаемого фото)
6. Нажмите **Queue Prompt**

**Результат:** 864x1080, префикс `instagram_flux`, папка `/content/ComfyUI/output/`

---

### Воркфлоу 2: `photo_flux_img2img` — Перенос стиля

Берёт **стиль и композицию** с референсного изображения и генерирует новое.

**Пошагово:**
1. Загрузите воркфлоу **photo_flux_img2img**
2. Загрузите референсное изображение (источник стиля/композиции)
3. Напишите промпт с описанием желаемого результата
4. Настройте `denoise` в KSampler:
   - `0.65` — умеренные изменения (ближе к оригиналу)
   - `0.80` — баланс между стилем и новизной
   - `0.90` — сильные изменения (свободная генерация)
5. Нажмите **Queue Prompt**

**Результат:** 864x1080, префикс `flux_img2img`, папка `/content/ComfyUI/output/`

---

### Использование LoRA (опционально)

Если у вас есть дообученная LoRA-модель на ваше лицо:
1. Загрузите файл LoRA в `/content/ComfyUI/models/loras/`
2. В ComfyUI добавьте ноду **LoraLoader** между CheckpointLoader и KSampler
3. Выберите файл вашей LoRA
4. Добавьте триггер-слово в промпт (например, `a photo of ohwx person`)
5. Рекомендуемая сила: `0.7`-`1.0`

---

### Пути к папкам (настроены для Colab)

| Папка | Путь | Назначение |
|-------|------|-----------|
| portrait | `/content/ComfyUI/input/refs/portrait` | Крупный план |
| hall | `/content/ComfyUI/input/refs/hall` | Интерьер |
| street | `/content/ComfyUI/input/refs/street` | Улица |
| output | `/content/ComfyUI/output/` | Результаты генерации |